# Tune `xgb_mspc`

MSPC + XGBoost. Repeated stratified CV on the train split; 
writes [`data/processed/tuned/xgb_mspc.json`](../data/processed/tuned/xgb_mspc.json).

**Stage 1 (hyperparameters):** PLS `n_components`, classifier `max_depth`, `learning_rate`. Select by **max mean PR AUC**.

**Stage 2 (threshold):** sweep `THRESHOLD_GRID` on the same CV folds; select threshold that **minimizes mean BER**.

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == "tuning" else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import scripts.tuning.registry as tuning_registry

importlib.reload(tuning_registry)
from scripts.secom_pipelines import TARGET_COL, feature_columns, load_mart, split_train_test
from scripts.tuning.registry import (
    MODEL_SPECS,
    fit_with_progress,
    run_grid_search,
    save_tuned_params,
    summarize_cv_search,
    tune_classifier_threshold,
    tuned_params_path,
)

MODEL_ID = "xgb_mspc"
spec = MODEL_SPECS[MODEL_ID]


In [2]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
print(len(X_train), "train rows", len(test_df), "test rows (holdout, not used here)")


1253 train rows 314 test rows (holdout, not used here)


In [3]:
param_grid = spec.make_param_grid()
pd.DataFrame([{k: v} for k, v in param_grid.items()])


,preprocess__sensor_mspc__pls__n_components,classifier__max_depth,classifier__learning_rate
0,"[15, 20, 25, 30]",NaN,NaN
1,NaN,"[4, 6, 8]",NaN
2,NaN,NaN,"[0.03, 0.05, 0.1]"


In [4]:
search, n_candidates, n_splits, total_fits = run_grid_search(spec, X_train, y_train)
print(f"{MODEL_ID}: {n_candidates} candidates x {n_splits} folds = {total_fits} fits")
search = fit_with_progress(search, X_train, y_train)


xgb_mspc: 36 candidates x 25 folds = 900 fits


GridSearchCV 900 fits:   0%|          | 0/900 [00:00<?, ?it/s]

  0%|          | 0/900 [00:00<?, ?it/s]

Fitting 25 folds for each of 36 candidates, totalling 900 fits


In [5]:
cv_summary, fold_results, aggregated = summarize_cv_search(search, spec)
print("Stage 1 best (mean PR AUC):")
display(aggregated.head(10))


Stage 1 best (mean PR AUC):


,n_components,max_depth,learning_rate,mean_ber_percent,std_ber_percent,mean_balanced_accuracy,mean_true_positive_percent,std_true_positive_percent,mean_true_negative_percent,std_true_negative_percent,mean_roc_auc,std_roc_auc,mean_pr_auc,std_pr_auc
24,25,8,0.03,45.113311,4.262043,0.548867,14.132353,8.397870,95.641026,1.226727,0.693678,0.071806,0.181868,0.060857
33,30,8,0.03,45.700352,3.950727,0.542996,12.941176,7.925502,95.658120,1.131825,0.695482,0.072394,0.181408,0.057658
27,30,4,0.03,44.397122,3.789401,0.556029,15.838235,7.587380,95.367521,1.051525,0.694650,0.065040,0.179642,0.057564
31,30,6,0.05,44.940234,4.156927,0.550598,14.632353,8.392022,95.487179,1.335523,0.689252,0.073242,0.179630,0.060356
4,15,6,0.05,44.735797,5.176104,0.552642,15.588235,10.094707,94.940171,1.265881,0.689938,0.066722,0.179518,0.061013
15,20,8,0.03,44.278281,4.398737,0.557217,16.058824,8.714225,95.384615,1.450475,0.699510,0.067128,0.179079,0.058340
30,30,6,0.03,44.914593,3.939816,0.550854,14.632353,7.836318,95.538462,1.172405,0.695848,0.070440,0.178629,0.057122
21,25,6,0.03,44.676911,4.413706,0.553231,15.073529,8.732372,95.572650,1.195848,0.691753,0.074401,0.178244,0.058441
6,15,8,0.03,44.907680,4.869880,0.550923,15.073529,9.779135,95.111111,1.399623,0.694363,0.066427,0.177985,0.061747
35,30,8,0.10,45.287016,3.806351,0.547130,13.955882,7.451167,95.470085,1.475442,0.683896,0.071804,0.177567,0.061531


In [6]:
threshold_result = tune_classifier_threshold(spec, X_train, y_train, cv_summary)
print(f"Stage 2 best threshold: {threshold_result['best_threshold']:.2f}")
print(f"  mean BER at threshold: {threshold_result['mean_ber_percent']:.2f}%")
display(threshold_result["per_threshold_mean_ber"].head(10))


Threshold CV folds:   0%|          | 0/25 [00:00<?, ?it/s]

Stage 2 best threshold: 0.05
  mean BER at threshold: 37.57%


,threshold,mean_ber_percent
0,0.05,37.570261
1,0.10,39.247612
2,0.15,40.802916
3,0.20,41.345337
4,0.25,42.139769
5,0.30,43.287959
6,0.35,43.992396
7,0.40,44.499309
8,0.45,44.881222
9,0.50,45.113311


In [7]:
payload = save_tuned_params(
    spec,
    cv_summary,
    fold_results,
    aggregated,
    threshold_result=threshold_result,
)
out_path = tuned_params_path(MODEL_ID)
print(f"Wrote {out_path}")
payload["grid_search_best_params"]


Wrote /home/troy/SECOM/data/processed/tuned/xgb_mspc.json


{'preprocess__sensor_mspc__pls__n_components': 25,
 'classifier__max_depth': 8,
 'classifier__learning_rate': 0.03}